In [122]:
# import necessary packages
import xarray as xr
import numpy as np
import pandas as pd
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(rc={'axes.facecolor': 'grey'})
plt.rcParams['figure.dpi'] = 300

In [123]:
os.chdir('C:/Users/edwin/OneDrive/Documents/GitHub/CHC')

In [124]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

list_of_files = glob.glob('data/csv/*.csv')
# Loop over all files
for f in list_of_files:
    # Generate the DataFrame
    df = pd.read_csv(f, sep=',', header=0, index_col=0)

    # Store the DataFrame in the dictionary with the year as key
    dfs_dict[f] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict.values(), ignore_index=True)

# Convert the 'date_of_prediction' column to datetime format
final_df['date_of_prediction'] = pd.to_datetime(final_df['date_of_prediction'])
final_df['month_of_prediction'] = final_df['date_of_prediction'].dt.month
final_df = final_df.drop(columns=['date_of_prediction', 'realization_year'])
final_df['precip'] = final_df['precip']/30

In [125]:
# Calculating all the statistics for each region and model
# groupby the region, model, season and month_of_prediction and calculate the corr between the predicted and actual precipitation for future use
corr = final_df.groupby(['region', 'model', 'season', 'month_of_prediction'])[['predicted_precip', 'precip']].corr(method = 'spearman').drop(['precip'], axis = 1).reset_index()
corr = corr.drop(corr.index[::2]).drop(columns = ['level_4'])
corr = corr.rename(columns = {'predicted_precip': 'corr'})

# Calculate mean and standard deviation
stat = final_df.groupby(['region', 'model', 'season', 'month_of_prediction']).agg(['mean', 'std']).reset_index()
stat.columns = ['region', 'model', 'season', 'month_of_prediction', 'pred_mean', 'pred_std', 'actual_mean', 'actual_std']

# Merging stat and spatial_means_corr to get 1 df with all values
stat_clean = stat.merge(corr, left_on=['region', 'model', 'season', 'month_of_prediction'], right_on=['region', 'model', 'season', 'month_of_prediction'], how='left').dropna()

# Calculating metrics
stat_clean['potential_skill'] = np.square(stat_clean['corr'])
stat_clean['conditional_bias'] = np.square(stat_clean['corr'] - (stat_clean['pred_std'] / stat_clean['actual_std']))
stat_clean['unconditional_bias'] = np.square((stat_clean['pred_mean'] - stat_clean['actual_mean']) / stat_clean['actual_std'])
stat_clean['skill_score'] = stat_clean['potential_skill'] - stat_clean['conditional_bias'] - stat_clean['unconditional_bias']

In [132]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='potential_skill')
    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }
    
    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]
    
    # Create the heatmap and force y ticklabels to remain visible
    ax = sns.heatmap(
        d,
        vmin=0, vmax=0.5,
        cmap=sns.color_palette('Reds', 10),
        fmt=".2f",
        linewidths=0.1,
        linecolor='black',
        square=True,
        yticklabels=True  # ensure ticklabels are drawn
    )
    
    # Set tick label properties explicitly
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=7)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)

    ax.invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=False, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Potential Skill by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/potential_skill.png')
plt.close()

In [26]:
stat_clean

,region,model,season,month_of_prediction,pred_mean,pred_std,actual_mean,actual_std,corr,potential_skill,conditional_bias,unconditional_bias,skill_score,region_season
0,eastern_east_africa,CCSM4,MAM,1,1.456676,0.180117,2.276974,0.562590,0.260264,0.067737,3.587119e-03,2.125987,-2.061836,eastern_east_africa | MAM
1,eastern_east_africa,CCSM4,MAM,2,1.488534,0.265555,2.276974,0.562590,0.374633,0.140350,9.484789e-03,1.964059,-1.833193,eastern_east_africa | MAM
2,eastern_east_africa,CCSM4,MAM,3,1.636503,0.345239,2.276974,0.562590,0.614003,0.377000,1.170352e-07,1.296031,-0.919031,eastern_east_africa | MAM
3,eastern_east_africa,CCSM4,MAM,9,1.165093,0.103833,2.276974,0.562590,0.052419,0.002748,1.746199e-02,3.906009,-3.920723,eastern_east_africa | MAM
4,eastern_east_africa,CCSM4,MAM,10,1.166946,0.129257,2.276974,0.562590,-0.005499,0.000030,5.534380e-02,3.893003,-3.948317,eastern_east_africa | MAM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1382,west_africa,SMME,JAS,3,7.184475,0.386880,7.368274,0.733531,0.461144,0.212654,4.392743e-03,0.062784,0.145476,west_africa | JAS
1383,west_africa,SMME,JAS,4,7.193658,0.373034,7.368274,0.733531,0.472874,0.223610,1.272435e-03,0.056667,0.165670,west_africa | JAS
1384,west_africa,SMME,JAS,5,7.438115,0.424314,7.368274,0.733531,0.379765,0.144222,3.947717e-02,0.009065,0.095679,west_africa | JAS
1385,west_africa,SMME,JAS,6,7.559975,0.464952,7.368274,0.733531,0.454912,0.206945,3.202032e-02,0.068298,0.106626,west_africa | JAS


In [43]:
# keep first 3 months of prediction of each model for each region and season
def keep_months_of_prediction(df, category):
    temp = df.copy()
    if category == 'low':
        for season in temp['season'].unique():
            if (temp[temp['season'] == season]['month_of_prediction'] == 12).any() \
            & (temp[temp['season'] == season]['month_of_prediction'] == 1).any():
                temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] = \
                temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] - 12
            # keep short lead (0-1) of prediction of each model for each region and season
            max = temp.loc[temp['season'] == season, 'month_of_prediction'].max()
            temp.loc[(temp['season'] == season)] = \
            temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= (max - 1))]
    elif category == 'medium':
        for season in temp['season'].unique():
            if (temp[temp['season'] == season]['month_of_prediction'] == 12).any() \
            & (temp[temp['season'] == season]['month_of_prediction'] == 1).any():
                temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] = \
                temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] - 12
            # keep medium category months (2-3) of prediction of each model for each region and season
            max = temp.loc[temp['season'] == season, 'month_of_prediction'].max()
            temp.loc[(temp['season'] == season)] = \
            temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] <= (max - 2)) & (temp['month_of_prediction'] >= (max - 3))]
    elif category == 'high':
        for season in temp['season'].unique():
            if (temp[temp['season'] == season]['month_of_prediction'] == 12).any() \
            & (temp[temp['season'] == season]['month_of_prediction'] == 1).any():
                temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] = \
                temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] - 12
            # keep last 3 months (4-6) of prediction of each model for each region and season
            max = temp.loc[temp['season'] == season, 'month_of_prediction'].max()
            temp.loc[(temp['season'] == season)] = \
            temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] <= (max - 4))]
    else:
        print("Invalid Category")
        return None
    return temp.dropna()


In [115]:
#potential_skill = keep_months_of_prediction(stat_clean[(stat_clean['model'] != 'SMME') & (stat_clean['model'] != 'MME')], 'low')
potential_skill = keep_months_of_prediction(stat_clean[(stat_clean['model'] != 'SMME') & (stat_clean['model'] != 'MME')], 'medium')
#potential_skill = keep_months_of_prediction(stat_clean[(stat_clean['model'] != 'SMME') & (stat_clean['model'] != 'MME')], 'high')

potential_skill = potential_skill.drop(columns = ['corr', 'conditional_bias', 'unconditional_bias', 'skill_score'])

# taking the mean of the month of prediction for each region and model and season
potential_skill = potential_skill.groupby(['region', 'model', 'season'])[['potential_skill']].mean().reset_index()

In [116]:
an = pd.read_csv('data/csv/metrics/region_model_df_high.csv', sep=',', header=0, index_col=0)
bn = pd.read_csv('data/csv/metrics/region_model_df_low.csv', sep=',', header=0, index_col=0)
an = an.rename(columns = {'agreement': 'an_agreement'})
bn = bn.rename(columns = {'agreement': 'bn_agreement'})

# taking the month of prediction mean for each region and model and season
an = an.groupby(['region', 'model', 'season'])[['an_agreement']].mean().reset_index()
bn = bn.groupby(['region', 'model', 'season'])[['bn_agreement']].mean().reset_index()

In [117]:
# merge an, bn and potential_skill on region, model 
an_bn = an.merge(bn, left_on=['region', 'model', 'season'], right_on=['region', 'model', 'season'], how='left')
an_bn = an_bn.dropna()

In [118]:
# SMME with hard cutoff
# merge potential_skill with an_bn on region, model and season
merged = potential_skill.merge(an_bn, left_on=['region', 'model', 'season'], right_on=['region', 'model', 'season'], how='left')
merged = merged.dropna()

# keep models with potential skill > 0.3 or (an_agreement > 0.4 and bn_agreement > 0.4)
merged = merged[(merged['potential_skill'] > 0.3) | ((merged['an_agreement'] > 0.4) & (merged['bn_agreement'] > 0.4))]
merged['region_season'] = merged['region'] + " | " + merged['season'] # compile region and season into one column, split by " | "
merged = merged.drop(columns = ['region', 'season'])

# create a nested dictionary where the keys are the regions, the values are lists of models
models = merged.groupby('region_season')['model'].apply(list).to_dict()
models

# separate the region and season into nested keys
SMME_models = {}
for region_season, model_list in models.items():
    region, season = region_season.split(" | ")
    if region not in SMME_models:
        SMME_models[region] = {}
    SMME_models[region][season] = model_list
print(SMME_models)

{'eastern_east_africa': {'MAM': ['CCSM4', 'CESM1', 'CMCC', 'CanESM5', 'DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA', 'NCEP'], 'OND': ['CMCC', 'CanESM5', 'DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA', 'NCEP']}, 'eastern_ukraine': {'JA': ['CMCC']}, 'lake_victoria_basin': {'MAM': ['CCSM4', 'CMCC', 'CanESM5', 'NASA', 'NCEP'], 'SON': ['GEM5', 'NCEP']}, 'south_sudan': {'ASO': ['NCEP'], 'JAS': ['DWD', 'GEM5', 'METEO'], 'MJJ': ['ECMWF', 'GFDL', 'NASA']}, 'southern_africa': {'FMA': ['DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'NASA']}, 'sri_lanka': {'OND': ['CESM1', 'CMCC', 'DWD', 'ECMWF', 'GEM5', 'GFDL', 'JMA', 'METEO', 'NASA', 'NCEP']}, 'west_africa': {'JAS': ['GFDL', 'NASA', 'NCEP']}}


In [119]:
#with open('SMME_models_low.txt', 'w') as f:
with open('SMME_models_medium.txt', 'w') as f:
#with open('SMME_models_high.txt', 'w') as f:
    for region, seasons in SMME_models.items():
        f.write(f"{region}:\n")
        for season, models in seasons.items():
            f.write(f"  {season}: {models}\n")  

In [99]:
# SMME with MME cutoff
merged_mod = potential_skill.merge(an_bn, left_on=['region', 'model', 'season'], right_on=['region', 'model', 'season'], how='left')
merged_mod = merged_mod.dropna()

In [100]:
# keep models that are higher than MME cutoff
MME_metrics = merged_mod[merged_mod['model'] != 'SMME']
MME_metrics['region_season'] = MME_metrics['region'] + " | " + MME_metrics['season'] # compile region and season into one column, split by " | "
MME_metrics = MME_metrics.drop(columns = ['region', 'season'])
MME_metrics = MME_metrics.groupby(['region_season'])[['model', 'potential_skill', 'an_agreement', 'bn_agreement']]




C:\Users\edwin\AppData\Local\Temp\ipykernel_20240\4248514773.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  MME_metrics['region_season'] = MME_metrics['region'] + " | " + MME_metrics['season'] # compile region and season into one column, split by " | "


In [101]:
SMME_dict = {}
for region_season, df in MME_metrics:
    region = region_season[0].split(" | ")[0]
    season = region_season[0].split(" | ")[1]
    potential_skill = df.loc[df['model'] == 'MME', 'potential_skill'].values[0]
    an_agreement = df.loc[df['model'] == 'MME', 'an_agreement'].values[0]
    bn_agreement = df.loc[df['model'] == 'MME', 'bn_agreement'].values[0]
    print(df)
    print(potential_skill)
    df = df[(df['potential_skill'] > potential_skill)]
    df.loc[:,'region_season'] = region_season[0]
    SMME_dict[region_season] = df

SMME_mod = pd.concat(SMME_dict.values(), ignore_index=True)
    

      model  potential_skill  an_agreement  bn_agreement
0     CCSM4         0.093756      0.480519      0.467532
2     CESM1         0.158559      0.544156      0.500000
4      CMCC         0.244237      0.583333      0.574074
6   CanESM5         0.235913      0.500000      0.568254
8       DWD         0.165803      0.492424      0.533333
10    ECMWF         0.136601      0.545455      0.454545
12     GEM5         0.217061      0.600000      0.533333
14     GFDL         0.235886      0.506494      0.584416
16      JMA         0.137327      0.496970      0.450000
18    METEO         0.115215      0.516667      0.600000
20      MME         0.248303      0.610390      0.610390
22     NASA         0.097957      0.519481      0.428571
24     NCEP         0.110227      0.467532      0.454545
0.2483027759417765


ValueError: cannot set a frame with no defined index and a scalar

In [76]:
SMME_mod

,model,potential_skill,an_agreement,bn_agreement
0,CanESM5,0.006104,0.383333,0.383333
1,NASA,0.014754,0.409091,0.378788
2,ECMWF,0.017192,0.409091,0.393939


In [120]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# initiate file list
list_of_files = glob.glob('data/csv/*.csv')

files_path = []

for path in list_of_files:
    path_mod = path.replace('\\', '/')
    files_path.append(path_mod)

for f in files_path:
    df = pd.read_csv(f)
    df['region'] = '_'.join(f.split('/')[-1].split('_')[0:-3])
    dfs_dict[f] = df

df = pd.concat(dfs_dict.values(), ignore_index=True)
smme_df = pd.DataFrame()
# keep only the rows with the models in SMME_models that matches with the keys in SMME_models
for region, seasons in SMME_models.items():
    for season, models in seasons.items():
        temp = df[(df['model'].isin(models)) & (df['region'] == region) & (df['season'] == season)]
        smme_df = pd.concat([smme_df, temp], ignore_index=True)
        
smme_df = smme_df[['region', 'season', 'date_of_prediction', 'realization_year', 'predicted_precip', 'precip']]\
                .groupby(['region', 'season', 'date_of_prediction', 'realization_year'])[['predicted_precip', 'precip']].mean().reset_index()
smme_df['model'] = 'SMME_medium'
smme_df

,region,season,date_of_prediction,realization_year,predicted_precip,precip,model
0,eastern_east_africa,MAM,1992-09-01,1993,2.014308,62.90869,SMME_medium
1,eastern_east_africa,MAM,1992-10-01,1993,1.908853,62.90869,SMME_medium
2,eastern_east_africa,MAM,1992-11-01,1993,2.013665,62.90869,SMME_medium
3,eastern_east_africa,MAM,1992-12-01,1993,2.179031,62.90869,SMME_medium
4,eastern_east_africa,MAM,1993-01-01,1993,2.162677,62.90869,SMME_medium
...,...,...,...,...,...,...,...
2399,west_africa,JAS,2024-03-01,2024,8.445797,255.36578,SMME_medium
2400,west_africa,JAS,2024-04-01,2024,8.788692,255.36578,SMME_medium
2401,west_africa,JAS,2024-05-01,2024,8.622224,255.36578,SMME_medium
2402,west_africa,JAS,2024-06-01,2024,9.148571,255.36578,SMME_medium


In [121]:
grouped = smme_df.groupby('region')
for region, region_df in grouped:
    region_name = str(region)
    model = 'SMME_medium'
    filename = f'data/csv/{region_name}_{model}_merged_seasonal.csv'
    region_df.to_csv(filename)

In [274]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='conditional_bias')

    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }

    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]

    sns.heatmap(d,
                vmin=0, vmax=1,
                cmap=sns.color_palette('Blues', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=False, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Conditional Bias by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/conditional_bias.png')
plt.close()

In [273]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='unconditional_bias')

    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }

    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]

    sns.heatmap(d,
                cmap=sns.color_palette('Blues', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=False, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Unconditional Bias by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/unconditional_bias.png')
plt.close()